In [1]:
from datetime import datetime
from pprint import pprint
from pathlib import Path
import sys

# Find repo root
repo_root = Path('.').resolve()
for _ in range(5):
    if (repo_root / 'shared_utils').is_dir():
        break
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

from capella import capella_v2
from shared_utils import s3utils
from shared_utils import cog_utils

Repo root: /home/jovyan/disasters-product-algorithms


In [2]:
###### CONFIG #######
bucket = "csdap-capellaspace-delivery"
prefix = "disasters"
upload_bucket = "s3://nasa-disasters/test_uploads"

# Target CRS for the output COG.
# None preserves the source projection (fastest, no warp).
# Uncomment the EPSG:3857 line if you plan to push the COG through
# veda-data-airflow's build_stac, which trips on the EPSG:4326 ensemble.
TARGET_CRS = None
# TARGET_CRS = "EPSG:3857"

In [3]:
tifs = capella_v2.retrieve_capella_resources("20260418193305", bucket, prefix)
pprint(tifs)

['s3://csdap-capellaspace-delivery/disasters/CAPELLA_C18_SM_GEO_HH_20260418193305_20260418193309/CAPELLA_C18_SM_GEO_HH_20260418193305_20260418193309.tif',
 's3://csdap-capellaspace-delivery/disasters/CAPELLA_C18_SM_SLC_HH_20260418193305_20260418193309/CAPELLA_C18_SM_SLC_HH_20260418193305_20260418193309.tif']


In [4]:
sig = capella_v2.sigmaCalib(tifs)
print(sig)

GEO file not found, downloading from S3
Generating Sigma Naught

	* Opening GEO File
DN range: 0 -> 65535
Scale factor: 0.0003869981737807393


/home/jovyan/disasters-product-algorithms/capella/capella_v2.py:145: RuntimeWarning: divide by zero encountered in log10
  sigma_0 = 20.0 * np.log10(scale_factor * dn)


Sigma0 range: -60.0 -> 28.083644387630564
Generation completed, file saved to ./s3_temp/202604_Capella-18_sigma02026-04-18T19:33:05Z.tif
./s3_temp/202604_Capella-18_sigma02026-04-18T19:33:05Z.tif


In [5]:
cog_filepath = cog_utils.convert_to_cog(sig, dst_crs=TARGET_CRS)
print(cog_filepath)
s3_upload_filepath = s3utils.build_flat_s3_uri(upload_bucket, cog_filepath.split("/")[-1])
print(s3_upload_filepath)
s3utils.upload_file_to_s3(cog_filepath, s3_upload_filepath)

  Auto-selected no-data value for float32: -9999.0
  Data type: float32
  No-data value: -9999.0
  Source CRS: EPSG:32655
  Compression: ZSTD (level 22)
  Overview levels: 5
  Creating COG: 202604_Capella-18_sigma02026-04-18T19:33:05Z.tif.cog.tmp.tif
  ✓ COG created: 202604_Capella-18_sigma02026-04-18T19:33:05Z.tif
./s3_temp/202604_Capella-18_sigma02026-04-18T19:33:05Z.tif
s3://nasa-disasters/test_uploads/202604_Capella-18_sigma02026-04-18T19:33:05Z.tif
Uploaded: s3://nasa-disasters/test_uploads/202604_Capella-18_sigma02026-04-18T19:33:05Z.tif


In [6]:
# Cleanup the s3_temp directory used for storing downloaded base geotiffs.
s3utils.remove_s3_temp()